<a href="https://colab.research.google.com/github/Mohd-Dilshan/ai-voice-sentiment-analyzer/blob/main/audio_sentiment_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# Install dependencies
!pip install openai-whisper
!apt install ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [14]:
# Import libraries
import whisper
from google.colab import files
from transformers import pipeline

# Load Whisper model
model = whisper.load_model("base")

# Upload audio file
uploaded = files.upload()

# Get file name
audio = list(uploaded.keys())[0]

# Transcribe audio
result = model.transcribe(audio)
transcribe = result["text"]

print("Transcription:")
print(transcribe)

# Sentiment analysis
sentiment = pipeline("sentiment-analysis")

sentiment_result = sentiment(transcribe)

print("Sentiment Result:")
print(sentiment_result)

Saving dilshan-audio.mp3 to dilshan-audio.mp3


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Transcription:
 Hello, I am Mohammed Dilshan. Today I want to share our tale about learning new technologies. At first it was confusing and sometimes frustrating because there were many concepts to understand. But I kept practicing everyday. Now things started clearer and more interesting.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Sentiment Result:
[{'label': 'POSITIVE', 'score': 0.9945662021636963}]


In [15]:
!pip install flask transformers pyngrok

In [16]:
!mkdir templates

mkdir: cannot create directory ‘templates’: File exists


In [17]:
%%writefile templates/index.html
<!DOCTYPE html>
<html>
<head>
<title>Audio Sentiment Analyzer</title>
</head>

<body>

<h2>Upload Audio File</h2>

<form action="/upload" method="post" enctype="multipart/form-data">

<input type="file" name="audio" required>

<br><br>

<button type="submit">Analyze Audio</button>

</form>

</body>
</html>

Overwriting templates/index.html


In [18]:
%%writefile templates/result.html
<!DOCTYPE html>
<html>
<head>
<title>Result</title>
</head>

<body>

<h2>Transcription</h2>
<p>{{ transcription }}</p>

<h2>Sentiment</h2>
<p>{{ sentiment }}</p>

<br>

<a href="/">Upload Another Audio</a>

</body>
</html>

Overwriting templates/result.html


In [19]:
%%writefile app.py

import whisper
from flask import Flask, request, render_template
from transformers import pipeline
import os

# Initialize Flask app
app = Flask(__name__)

# Load models
model = whisper.load_model("base")
sentiment = pipeline("sentiment-analysis")

# Home page
@app.route("/")
def home():
    return render_template("index.html")

# Upload route
@app.route("/upload", methods=["POST"])
def upload():

    file = request.files["audio"]

    filename = file.filename
    filepath = os.path.join(filename)

    file.save(filepath)

    # Transcribe audio
    result = model.transcribe(filepath)
    text = result["text"]

    # Sentiment analysis
    sentiment_result = sentiment(text)

    return render_template(
        "result.html",
        transcription=text,
        sentiment=sentiment_result
    )

if __name__ == "__main__":
    app.run()

Overwriting app.py


In [20]:
from pyngrok import ngrok
import subprocess

In [21]:
!pip install gradio

In [22]:
import whisper
from transformers import pipeline
import gradio as gr

# Load models
model = whisper.load_model("base")
sentiment = pipeline("sentiment-analysis")

# Function to process audio
def analyze_audio(audio):

    result = model.transcribe(audio)
    text = result["text"]

    sentiment_result = sentiment(text)

    return text, sentiment_result

# Create web interface
app = gr.Interface(
    fn=analyze_audio,
    inputs=gr.Audio(type="filepath"),
    outputs=["text","text"],
    title="Audio Sentiment Analyzer"
)

# Launch web app
app.launch(share=True)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f9e054fcc50661ab5a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
